# Topic 4: select, filter, withColumn — Core DataFrame Transformations

> **Phase 3 → Week 3 → Topic 4**

---

## The Spreadsheet Analogy

Think of a DataFrame as a spreadsheet:
- `select` = choose which columns to show (hide others)
- `filter` = turn on AutoFilter to show only matching rows
- `withColumn` = add a new computed column (formula column)
- `withColumnRenamed` = rename a column header
- `drop` = hide a column permanently
- `cast` = change the data type of a cell range
- `when/otherwise` = `=IF(condition, value_if_true, value_if_false)`

---

## The `col()` Function — Your Most-Used Tool

`F.col("column_name")` creates a **Column expression** — the building block of most DataFrame operations.

```python
from pyspark.sql import functions as F

# These are equivalent ways to reference a column:
df["salary"]             # dict-style
df.salary                # attribute-style (breaks for names with spaces)
F.col("salary")          # col() — most explicit, preferred in functions
F.expr("salary * 0.3")   # SQL expression string — for complex expressions
```

**Use `F.col()` in your code.** It's the clearest and most flexible.

---

## `when` / `otherwise` — The CASE WHEN of PySpark

```python
# SQL equivalent:
# CASE WHEN salary > 90000 THEN 'Senior'
#      WHEN salary > 70000 THEN 'Mid'
#      ELSE 'Junior'
# END

F.when(F.col("salary") > 90000, "Senior") \
 .when(F.col("salary") > 70000, "Mid") \
 .otherwise("Junior")
```

Chain as many `.when()` as needed, end with `.otherwise()`. Without `.otherwise()`, non-matching rows get `null`.

---

## Interview Cheat Sheet

**Q: What is the difference between `filter()` and `where()` in Spark?**
> They are identical — `where()` is just an alias for `filter()`. Both accept a Column expression or a SQL string. Use whichever reads more naturally: DataFrame fans use `filter()`, SQL fans use `where()`.

**Q: What is the difference between `select()` and `withColumn()`?**
> `select()` returns a new DataFrame with ONLY the specified columns (drops all others). `withColumn()` adds or replaces ONE column while keeping ALL existing columns. Use `select()` when you want to reshape/reduce the schema; use `withColumn()` when you want to add a derived column.

**Q: What does `F.lit()` do?**
> `F.lit(value)` creates a Column containing a literal constant value — the same value for every row. Example: `df.withColumn("source", F.lit("etl_pipeline"))` adds a column where every row has the string `"etl_pipeline"`. Needed because column expressions expect Column objects, not Python scalars.

**Q: What is `F.expr()` and when would you use it?**
> `F.expr(sql_string)` evaluates a SQL expression string as a Column. Use it for complex expressions that are easier to write in SQL than in the functional API: `F.expr("salary * 0.3 + bonus")` or `F.expr("CASE WHEN salary > 100000 THEN 'high' ELSE 'low' END")`.

**Q: How do you handle null values in a filter?**
> `null` comparisons always return `null` (not True or False) in SQL/Spark. Use `isNull()` or `isNotNull()` explicitly: `df.filter(F.col("name").isNotNull())`. Never use `== None` — it won't work as expected.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date

spark = SparkSession.builder \
    .appName("Week3 - select filter withColumn") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

# Load employees dataset
schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("name",      StringType(),  True),
    StructField("dept",      StringType(),  True),
    StructField("salary",    IntegerType(), True),
    StructField("country",   StringType(),  True),
    StructField("join_date", DateType(),    True),
    StructField("active",    BooleanType(), True),
])

df = spark.read.schema(schema).option("header", True) \
    .option("dateFormat", "yyyy-MM-dd").csv("week3/data/employees.csv")

print("Dataset loaded:")
df.show()

## Part 1: `select()` — Choose Columns

In [ ]:
# Basic select — choose columns by name
print("=== select() ===")

# Method 1: string names
df.select("name", "dept", "salary").show()

# Method 2: col() expressions — allows aliasing and transformations
df.select(
    F.col("name"),
    F.col("dept"),
    F.col("salary"),
    (F.col("salary") * 0.3).alias("tax"),           # computed column
    F.col("join_date").alias("start_date"),          # rename inline
).show()

# Method 3: expr() — SQL expressions
df.select(
    "name",
    F.expr("salary * 0.3 AS tax"),
    F.expr("UPPER(dept) AS dept_upper"),
).show()

In [ ]:
# Select ALL columns + extras using * trick
df.select("*", (F.col("salary") * 12).alias("annual_salary")).show(3)

# Select by excluding columns
all_except_id = [c for c in df.columns if c != "id"]
df.select(all_except_id).show(3)

# selectExpr — pure SQL strings, very concise
df.selectExpr(
    "name",
    "salary",
    "salary * 0.3 AS tax",
    "UPPER(dept) AS dept",
    "DATEDIFF(CURRENT_DATE(), join_date) AS days_employed",
).show()

## Part 2: `filter()` / `where()` — Filter Rows

In [ ]:
print("=== filter() / where() ===")

# Single condition
df.filter(F.col("salary") > 80000).show()

# Multiple conditions — AND / OR
df.filter(
    (F.col("salary") > 70000) & (F.col("country") == "India")
).show()

df.filter(
    (F.col("dept") == "Engineering") | (F.col("salary") > 90000)
).show()

# NOT condition
df.filter(~(F.col("dept") == "Sales")).show(3)

# SQL string condition — great for complex expressions
df.filter("salary > 70000 AND country = 'India'").show()
df.where("dept IN ('Engineering', 'Marketing')").show()

In [ ]:
# String filters — startsWith, endsWith, contains, like, rlike
df.filter(F.col("name").startsWith("A")).show()
df.filter(F.col("name").endsWith("e")).show()
df.filter(F.col("name").contains("a")).show()
df.filter(F.col("dept").like("%ing%")).show()       # SQL LIKE (% = wildcard)
df.filter(F.col("name").rlike("^[AC]")).show()      # regex — names starting with A or C

# isin — IN list
df.filter(F.col("country").isin(["India", "USA"])).show()
df.filter(~F.col("country").isin(["India", "USA"])).show()  # NOT IN

# between
df.filter(F.col("salary").between(60000, 90000)).show()

In [ ]:
# NULL handling in filters
null_df = spark.createDataFrame([
    (1, "Alice",  95000),
    (2, None,     88000),
    (3, "Carol",  None),
], ["id", "name", "salary"])

print("=== NULL handling in filter ===")

# WRONG — never use == None
print("df.filter(col('name') == None) — WRONG, returns empty:")
null_df.filter(F.col("name") == None).show()   # returns empty!

# CORRECT — use isNull()
print("df.filter(col('name').isNull()) — CORRECT:")
null_df.filter(F.col("name").isNull()).show()
null_df.filter(F.col("name").isNotNull()).show()

# Filter for actual values (skip nulls automatically)
null_df.filter(F.col("salary") > 90000).show()   # nulls are excluded automatically

## Part 3: `withColumn()` — Add or Replace a Column

In [ ]:
print("=== withColumn() ===")

# Add a new column
df2 = df.withColumn("tax",          F.col("salary") * 0.3) \
         .withColumn("net_salary",   F.col("salary") - F.col("salary") * 0.3) \
         .withColumn("senior_flag",  F.col("salary") > 90000) \
         .withColumn("source",       F.lit("hr_system"))     # literal constant

df2.select("name", "salary", "tax", "net_salary", "senior_flag", "source").show()

# Replace an existing column (same column name)
df3 = df.withColumn("salary", F.col("salary") * 1.10)  # 10% raise!
print("After 10% raise:")
df3.select("name", "salary").show()

In [ ]:
# when / otherwise — CASE WHEN
print("=== when() / otherwise() — CASE WHEN ===")

df_graded = df.withColumn(
    "salary_grade",
    F.when(F.col("salary") > 90000, "Senior")
     .when(F.col("salary") > 70000, "Mid")
     .when(F.col("salary") > 55000, "Junior")
     .otherwise("Entry")
)
df_graded.select("name", "salary", "salary_grade").show()

# when based on multiple conditions
df.withColumn(
    "highlight",
    F.when(
        (F.col("salary") > 80000) & (F.col("country") == "India"),
        "High earner in India"
    ).when(
        F.col("active") == False,
        "Inactive"
    ).otherwise("Normal")
).select("name", "salary", "country", "active", "highlight").show()

In [ ]:
# Date functions with withColumn — very common in ETL
print("=== Date operations with withColumn ===")

df_dates = df.withColumn("year_joined",     F.year("join_date")) \
              .withColumn("month_joined",    F.month("join_date")) \
              .withColumn("days_employed",   F.datediff(F.current_date(), F.col("join_date"))) \
              .withColumn("years_employed",  F.round(F.datediff(F.current_date(), F.col("join_date")) / 365, 1)) \
              .withColumn("is_senior",       F.col("days_employed") > 365 * 3)

df_dates.select("name", "join_date", "year_joined", "days_employed", "years_employed", "is_senior").show()

## Part 4: `withColumnRenamed`, `drop`, `alias`, `cast`

In [ ]:
# withColumnRenamed — rename a column
df.withColumnRenamed("dept", "department") \
  .withColumnRenamed("join_date", "start_date") \
  .show(3)

# Rename multiple columns at once (Python loop)
rename_map = {"dept": "department", "join_date": "start_date", "active": "is_active"}
df_renamed = df
for old, new in rename_map.items():
    df_renamed = df_renamed.withColumnRenamed(old, new)
df_renamed.show(3)

In [ ]:
# drop — remove columns
print("drop():")
df.drop("active", "join_date").show(3)

# cast — change data type
print("cast():")
df.withColumn("salary_str",  F.col("salary").cast(StringType())) \
  .withColumn("salary_dbl",  F.col("salary").cast("double")) \
  .withColumn("salary_long", F.col("salary").cast(LongType())) \
  .select("name", "salary", "salary_str", "salary_dbl", "salary_long").show(3)

# alias — rename a column expression
df.select(
    F.col("name").alias("employee_name"),
    (F.col("salary") / 12).alias("monthly_salary"),
).show(3)

## Part 5: `F.lit()`, `F.col()`, `F.expr()` — The Column Building Blocks

In [ ]:
print("=== F.lit() — Literal constant column ===")

df.withColumn("currency",   F.lit("INR")) \
  .withColumn("loaded_at",  F.lit(date(2024, 1, 15)).cast(DateType())) \
  .withColumn("pipeline_v", F.lit(2)) \
  .select("name", "salary", "currency", "loaded_at", "pipeline_v").show(3)

print("\n=== F.expr() — Complex SQL expressions ===")

df.select(
    "name",
    "salary",
    F.expr("salary * 0.3").alias("tax"),
    F.expr("CASE WHEN salary > 90000 THEN 'Senior' ELSE 'Junior' END").alias("grade"),
    F.expr("UPPER(name) || ' - ' || dept").alias("full_label"),
    F.expr("DATEDIFF(CURRENT_DATE, join_date) / 365").alias("years_worked"),
).show()

## Part 6: Common String Functions

In [ ]:
print("=== String Functions ===")

df.select(
    F.col("name"),
    F.upper("name").alias("upper"),
    F.lower("name").alias("lower"),
    F.length("name").alias("len"),
    F.substring("name", 1, 3).alias("first3"),   # 1-indexed!
    F.trim("name").alias("trimmed"),
    F.concat(F.col("name"), F.lit(" @ "), F.col("dept")).alias("label"),
    F.regexp_replace("name", "[aeiou]", "*").alias("masked_vowels"),
    F.split("dept", "i").alias("dept_split"),
).show(truncate=False)

## Exercises

1. From employees: select only Engineering employees in India earning over 70k. Show name, salary, join_date.
2. Add three columns: `monthly_salary`, `tax_30pct`, `net_monthly`. Chain all with `withColumn`.
3. Add a `seniority` column using `when/otherwise`: Senior if >5 years, Mid if >2 years, Junior otherwise (use `days_employed / 365`).
4. Use `selectExpr` to create a single-shot transformed DataFrame: uppercase name, salary in thousands, department abbreviated to first 3 chars.

In [ ]:
# Exercise 1
print("Exercise 1 — Engineering, India, salary > 70k:")
df.filter(
    (F.col("dept") == "Engineering") &
    (F.col("country") == "India") &
    (F.col("salary") > 70000)
).select("name", "salary", "join_date").show()

In [ ]:
# Exercise 3 — seniority with when/otherwise
print("Exercise 3 — Seniority column:")
df.withColumn(
    "days", F.datediff(F.current_date(), F.col("join_date"))
).withColumn(
    "seniority",
    F.when(F.col("days") > 365 * 5, "Senior")
     .when(F.col("days") > 365 * 2, "Mid")
     .otherwise("Junior")
).select("name", "join_date", "days", "seniority").show()

In [ ]:
# Exercise 4 — selectExpr one-shot transform
print("Exercise 4 — selectExpr one-shot:")
df.selectExpr(
    "UPPER(name) AS name",
    "ROUND(salary / 1000, 1) AS salary_k",
    "SUBSTRING(dept, 1, 3) AS dept_abbr",
    "country",
).show()